<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Получение-и-обработка-данных-о-границах-районов" data-toc-modified-id="Получение-и-обработка-данных-о-границах-районов-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Получение и обработка данных о границах районов</a></span><ul class="toc-item"><li><span><a href="#Источник-данных" data-toc-modified-id="Источник-данных-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Источник данных</a></span></li><li><span><a href="#Запрос-на-границы-районов-Москвы" data-toc-modified-id="Запрос-на-границы-районов-Москвы-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Запрос на границы районов Москвы</a></span></li></ul></li><li><span><a href="#Получение-данных-о-поездках-в-такси-и-каршеринге-через-API" data-toc-modified-id="Получение-данных-о-поездках-в-такси-и-каршеринге-через-API-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Получение данных о поездках в такси и каршеринге через API</a></span></li><li><span><a href="#Обработка-данных" data-toc-modified-id="Обработка-данных-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Обработка данных</a></span></li><li><span><a href="#Добавление-координат" data-toc-modified-id="Добавление-координат-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Добавление координат</a></span></li><li><span><a href="#Добавление-данных-о-погоде" data-toc-modified-id="Добавление-данных-о-погоде-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Добавление данных о погоде</a></span></li><li><span><a href="#Обработка-названий-полей-и-пропусков-в-погодных-данных" data-toc-modified-id="Обработка-названий-полей-и-пропусков-в-погодных-данных-6"><span class="toc-item-num">6&nbsp;&nbsp;</span>Обработка названий полей и пропусков в погодных данных</a></span></li><li><span><a href="#Выделение-новых-временных-признаков" data-toc-modified-id="Выделение-новых-временных-признаков-7"><span class="toc-item-num">7&nbsp;&nbsp;</span>Выделение новых временных признаков</a></span></li><li><span><a href="#Добавление-данных-о-загруженности-метро" data-toc-modified-id="Добавление-данных-о-загруженности-метро-8"><span class="toc-item-num">8&nbsp;&nbsp;</span>Добавление данных о загруженности метро</a></span></li><li><span><a href="#Развлекательные-места-в-радиусе-5-км" data-toc-modified-id="Развлекательные-места-в-радиусе-5-км-9"><span class="toc-item-num">9&nbsp;&nbsp;</span>Развлекательные места в радиусе 5 км</a></span></li><li><span><a href="#Добавление-информации-о-населении-районов-и-доступности-метро" data-toc-modified-id="Добавление-информации-о-населении-районов-и-доступности-метро-10"><span class="toc-item-num">10&nbsp;&nbsp;</span>Добавление информации о населении районов и доступности метро</a></span></li><li><span><a href="#Описание-полей-(неполное)" data-toc-modified-id="Описание-полей-(неполное)-11"><span class="toc-item-num">11&nbsp;&nbsp;</span>Описание полей (неполное)</a></span><ul class="toc-item"><li><span><a href="#Описание-полей-DataFrame-о-поездках-такси-и-каршеринга-по-районам-Москвы" data-toc-modified-id="Описание-полей-DataFrame-о-поездках-такси-и-каршеринга-по-районам-Москвы-11.1"><span class="toc-item-num">11.1&nbsp;&nbsp;</span>Описание полей DataFrame о поездках такси и каршеринга по районам Москвы</a></span></li></ul></li></ul></div>

In [1]:
import json
import pandas as pd
import requests
import time
import warnings

from tqdm import tqdm
from datetime import datetime
from meteostat import Hourly, Point

warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'meteostat'

## Получение и обработка данных о границах районов

### Источник данных
**Источник:** [Overpass Turbo](https://overpass-turbo.eu/)

---
### Запрос на границы районов Москвы
```overpass
[out:json];
area[name="Москва"]->.boundary;
(
  relation["boundary"="administrative"]["admin_level"="8"](area.boundary);
);
out geom;
```

Выполняем запрос, затем делаем экспорт данных как GeoJSON, вручную меняем название на export_borders.geojson, сохраняем в рабочую директорию

In [ ]:
path_borders = 'https://raw.githubusercontent.com/RuslanDavletov/Analysis-of-taxi-demand/main/data/export_borders.geojson'
path_centers = 'https://raw.githubusercontent.com/RuslanDavletov/Analysis-of-taxi-demand/main/data/export_centers.geojson'
path_stations = "https://raw.githubusercontent.com/RuslanDavletov/Analysis-of-taxi-demand/main/data/station_load_data_res%20(1).csv"

In [ ]:
response = requests.get(path_borders)
data = response.json()

In [ ]:
data

In [ ]:
data_district = []

# Перебираем районы
for feature in data['features']:
    properties = feature['properties']
    geometry = feature['geometry']

    if geometry['type'] == 'MultiPolygon':
        for polygon in geometry['coordinates']:
            coords = [{'lng': point[0], 'lat': point[1]} for point in polygon[0]]
            district = {
                "type": "Polygon",
                "coordinates": [coords],
                "name": properties.get('name', 'Без названия'),
                "id": properties.get('@id', 'Неизвестно')
            }
            data_district.append(district)

    elif geometry['type'] == 'Polygon':
        coords = [{'lng': point[0], 'lat': point[1]} for point in geometry['coordinates'][0]]
        district = {
            "type": "Polygon",
            "coordinates": [coords],
            "name": properties.get('name', 'Без названия'),
            "id": properties.get('@id', 'Неизвестно')
        }
        data_district.append(district)


In [ ]:
data_district

## Получение данных о поездках в такси и каршеринге через API

Источник - https://prodvizhenie.mos.ru

In [ ]:
url = 'https://prodvizhenie.mos.ru/api/graphql'

headers = {
    'Content-Type': 'application/json',
    'User-Agent': 'Mozilla/5.0',
}

query = """
query GeometryStatsMapTooltip(
  $customGeometry: [[[Point!]]],
  $filter: Filter!,
  $source: DataSourceType!,
  $eventType: EventType!
) {
  currentPeriod: ridePolygonStats(
    ids: []
    customGeometry: $customGeometry
    filter: $filter
    source: $source
    aggregation: GEOMETRY
    eventType: $eventType
  ) {
    totalRideCount
    data {
      date
      hour
      rideCount
      __typename
    }
    __typename
  }
}
"""

In [ ]:
def get_taxi_data(fromDate, toDate, coords, source, eventType):
    variables = {
        "filter": {
            "fromDate": fromDate,
            "toDate": toDate,
            "hours": [0, 23],
            "weekdays": [1, 2, 3, 4, 5, 6, 7]
        },
        "customGeometry": [coords],
        "source": f"{source}",
        "eventType": f"{eventType}"
    }

    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    data = response.json()

    # Оставляем только полезные данные
    result = data.get('data', {}).get('currentPeriod', {})
    return {
        "totalRideCount": result.get('totalRideCount', 0),
        "rides": result.get('data', [])
    }

In [ ]:
def get_df(start_date, end_date, source, event):
    main_df = pd.DataFrame()

    for i in tqdm(range(len(data_district)), desc="Обработка районов"):
        coords = data_district[i]['coordinates']
        get_json = get_taxi_data(start_date, end_date, coords, source, event)

        # Преобразование в DataFrame
        df = pd.DataFrame(get_json["rides"])

        # Добавим столбцы с информацией о районе
        df['district_name'] = data_district[i]['name']
        df['district_id'] = i

        main_df = pd.concat([main_df, df])

        # Задержка в 1 секунду
        time.sleep(1)
    return main_df


In [ ]:
taxi_start = get_df("2024-01-01", "2025-01-01", 'TAXI', 'START')

In [ ]:
taxi_finish = get_df("2024-01-01", "2025-01-01", 'TAXI', 'FINISH')

In [ ]:
carsharing_start = get_df("2024-01-01", "2025-01-01", 'CARSHARING', 'START')

In [ ]:
carsharing_finish = get_df("2024-01-01", "2025-01-01", 'CARSHARING', 'FINISH')

## Обработка данных

In [ ]:
taxi_start.rename(columns={'rideCount': 'taxi_start_rideCount'}, inplace=True)
taxi_finish.rename(columns={'rideCount': 'taxi_finish_rideCount'}, inplace=True)
carsharing_start.rename(columns={'rideCount': 'carsharing_start_rideCount'}, inplace=True)
carsharing_finish.rename(columns={'rideCount': 'carsharing_finish_rideCount'}, inplace=True)

In [ ]:
taxi_start.drop(columns=['__typename'], inplace=True)
taxi_finish.drop(columns=['__typename'], inplace=True)
carsharing_start.drop(columns=['__typename'], inplace=True)
carsharing_finish.drop(columns=['__typename'], inplace=True)

In [ ]:
taxi_start['timestamp'] = pd.to_datetime(taxi_start['date']) + pd.to_timedelta(taxi_start['hour'], unit='h')
taxi_start.drop(columns=['date', 'hour'], inplace=True)

taxi_finish['timestamp'] = pd.to_datetime(taxi_finish['date']) + pd.to_timedelta(taxi_finish['hour'], unit='h')
taxi_finish.drop(columns=['date', 'hour'], inplace=True)

carsharing_start['timestamp'] = pd.to_datetime(carsharing_start['date']) + pd.to_timedelta(carsharing_start['hour'], unit='h')
carsharing_start.drop(columns=['date', 'hour'], inplace=True)

carsharing_finish['timestamp'] = pd.to_datetime(carsharing_finish['date']) + pd.to_timedelta(carsharing_finish['hour'], unit='h')
carsharing_finish.drop(columns=['date', 'hour'], inplace=True)

In [ ]:
carsharing_finish.sample(5)

In [ ]:
main_df = taxi_start.merge(taxi_finish, on=['district_id', 'district_name', 'timestamp'], how='outer')

In [ ]:
main_df = main_df.merge(carsharing_start, on=['district_id', 'district_name', 'timestamp'], how='outer')

In [ ]:
main_df = main_df.merge(carsharing_finish, on=['district_id', 'district_name', 'timestamp'], how='outer')

In [ ]:
main_df.head()

In [ ]:
main_df[['taxi_start_rideCount', 'taxi_finish_rideCount', 'carsharing_start_rideCount', 'carsharing_finish_rideCount']] = main_df[['taxi_start_rideCount', 'taxi_finish_rideCount', 'carsharing_start_rideCount', 'carsharing_finish_rideCount']].fillna(0)

In [ ]:
main_df.groupby(['district_id']).timestamp.count()

In [ ]:
main_df.info()

In [ ]:
main_df['timestamp'] = pd.to_datetime(main_df['timestamp'])

full_range = pd.date_range(start=main_df['timestamp'].min(),
                           end=main_df['timestamp'].max(),
                           freq='H')

districts = main_df[['district_name', 'district_id']].drop_duplicates()

full_df = (districts.assign(key=1)
           .merge(pd.DataFrame({'timestamp': full_range, 'key': 1}), on='key')
           .drop(columns='key'))

df_filled = (full_df.merge(main_df,
                           on=['timestamp', 'district_name', 'district_id'],
                           how='left')
                    .fillna(0))


In [ ]:
df_filled

In [ ]:
df_filled.info()

In [ ]:
# Проверка что количество регионов бьется с википедией

df_filled.district_name.nunique()

## Добавление координат

Источник данных
**Источник:** [Overpass Turbo](https://overpass-turbo.eu/)

---
Запрос на центры районов Москвы
```overpass
[out:json];
area[name="Москва"]->.moscow;
relation["admin_level"="8"](area.moscow);
out center tags;
```
Экспорт geojson, название файла - export_centers.geojson

In [ ]:
df_filled.head()

In [ ]:
response_centeres = requests.get(path_centers)
data_centeres = response_centeres.json()

df = pd.json_normalize(data_centeres['features'])
df.head()

In [ ]:
df.columns

In [ ]:
df_main = df[['properties.name', 'geometry.coordinates']]

In [ ]:
df_main = df_main.copy()
df_main['lat'] = df_main['geometry.coordinates'].apply(lambda x: x[1])
df_main['lon'] = df_main['geometry.coordinates'].apply(lambda x: x[0])
df_main.drop(columns=['geometry.coordinates'], inplace=True)

In [ ]:
df_main.head()

In [ ]:
df_main.columns = ['district_name', 'lat', 'lon']

In [ ]:
df_with_coords = df_filled.merge(df_main, how='left', on='district_name')

In [ ]:
df_with_coords.sample(10)

## Добавление данных о погоде

In [ ]:
df_with_coords.head()

In [ ]:
df_with_coords.info()

In [ ]:
# Переводим столбец timestamp в формат datetime
df_with_coords["timestamp"] = pd.to_datetime(df_with_coords["timestamp"])

# Получаем уникальные пары координат
unique_points = df_with_coords[["lat", "lon"]].drop_duplicates()

# Словарь для хранения погодных данных по координатам
weather_data_all = []

start = df_with_coords["timestamp"].min()
end = df_with_coords["timestamp"].max()

# Получаем погодные данные для каждой уникальной точки
for _, row in tqdm(unique_points.iterrows(), total=len(unique_points)):
    point = Point(row["lat"], row["lon"])
    data = Hourly(point, start, end).fetch().reset_index()

    # Добавляем координаты для последующего объединения
    data["lat"] = row["lat"]
    data["lon"] = row["lon"]

    # Оставляем только нужные столбцы
    data = data[["time", "lat", "lon", "temp", "prcp", "rhum", "wspd", "coco"]]
    data = data.rename(columns={"time": "timestamp"})

    weather_data_all.append(data)

# Объединяем все погодные данные в один датафрейм
weather_df = pd.concat(weather_data_all, ignore_index=True)

In [ ]:
weather_df

In [ ]:
ready_df = df_with_coords.merge(weather_df, how='left', on=["timestamp", "lat", "lon"])


## Обработка названий полей и пропусков в погодных данных

In [ ]:
ready_df.rename(columns={'temp': 'temperature',
                         'prcp': 'precipitation',
                         'rhum': 'humidity',
                         'wspd': 'wind_speed',
                         'taxi_start_rideCount': 'n_taxi_start',
                         'taxi_finish_rideCount': 'n_taxi_end',
                         'carsharing_start_rideCount': 'n_carsharing_start',
                         'carsharing_finish_rideCount': 'n_carsharing_end',
                        'coco': 'weather_code'
                        }, inplace=True)

In [ ]:
# задал порядок полей

ready_df = ready_df[['timestamp', 'district_name', 'district_id', 'lat', 'lon', 'n_taxi_start', 'n_taxi_end', \
'n_carsharing_start','n_carsharing_end', 'temperature','precipitation','humidity','wind_speed','weather_code']]

In [ ]:
ready_df.sample(10)

In [ ]:
ready_df.isnull().sum()

In [ ]:
# Пропуски были по timestamp, а не по координатам, поэтому заполнил последним известным precipitation
ready_df['precipitation'] = ready_df['precipitation'].ffill()

In [ ]:
ready_df['weather_code'] = ready_df['weather_code'].ffill()

In [ ]:
# ready_df.to_csv('main_df.csv', index=False)

## Выделение новых временных признаков

In [ ]:
ready_df.head()

In [ ]:
# Категориальный признак утренний / вечерний час пик
def classify_rush_hour(timestamp):
    hour = timestamp.hour
    if 7 <= hour < 10:
        return 1
    elif 17 <= hour < 20:
        return 2
    else:
        return 0

In [ ]:
ready_df['rush_hour'] = ready_df['timestamp'].apply(classify_rush_hour)

In [ ]:
def get_season(timestamp):
    month = timestamp.month
    if month in [12, 1, 2]:
        return 0  # Зима
    elif month in [3, 4, 5]:
        return 1  # Весна
    elif month in [6, 7, 8]:
        return 2  # Лето
    else:
        return 3  # Осень

In [ ]:
ready_df['season'] = ready_df['timestamp'].apply(get_season)

In [ ]:
# Убрал 1 января 2025, чтобы не было ошибок с другими признаками

ready_df = ready_df[ready_df['timestamp'].dt.year == 2024]

In [ ]:
# Российские праздники
russian_holidays = [
    "2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-06", "2024-01-07", "2024-01-08",  # Новогодние праздники и Рождество
    "2024-02-23",  # День защитника Отечества
    "2024-03-08",  # Международный женский день
    "2024-05-01",  # Праздник Весны и Труда
    "2024-05-09",  # День Победы
    "2024-05-10",  # День Победы
    "2024-06-12",  # День России
    "2024-11-04",  # Новогодние праздники
    "2024-12-30",  # Новогодние праздники
    "2024-12-31",  # Новогодние праздники
]
russian_holidays = pd.to_datetime(russian_holidays)

In [ ]:
def is_holiday_or_weekend(timestamp):
    return 1 if (timestamp.weekday() >= 5 or timestamp.date() in russian_holidays.date) else 0

In [ ]:
ready_df['is_holiday_or_weekend'] = ready_df['timestamp'].apply(is_holiday_or_weekend)

In [ ]:
ready_df.sample(5)

In [ ]:
# ready_df.to_csv('main_df.csv', index=False)

## Добавление данных о загруженности метро

In [ ]:
import pandas as pd
passazhiro_potok = 'https://raw.githubusercontent.com/RuslanDavletov/Analysis-of-taxi-demand/refs/heads/main/data/passazhiro_potok.csv'

In [ ]:
data = pd.read_csv(passazhiro_potok, delimiter=';')

In [ ]:
data

In [ ]:
data = data.drop(index=0)
data = data.drop(columns=["Unnamed: 7"])
data.head()

In [ ]:
data.dtypes

Перевожу числовые данные в нормальный вид

In [ ]:
data["Year"] = pd.to_numeric(data["Year"], errors="coerce")
data["IncomingPassengers"] = pd.to_numeric(data["IncomingPassengers"], errors="coerce")
data["OutgoingPassengers"] = pd.to_numeric(data["OutgoingPassengers"], errors="coerce")

data.dtypes

Логичнее будет смотреть на загруженность по году и кварталу одновременно

In [ ]:
df = data.groupby(['Year', 'Quarter', "NameOfStation"])[['IncomingPassengers', 'OutgoingPassengers']].apply(lambda x: x['IncomingPassengers'] - x['OutgoingPassengers'])
df

In [ ]:
grouped = data.groupby(['Year', 'Quarter', 'NameOfStation', 'Line']).agg(IncomingPassengers=('IncomingPassengers', 'sum'),
    OutgoingPassengers=('OutgoingPassengers', 'sum'))

grouped['PassengerDifference'] = grouped['IncomingPassengers'] - grouped['OutgoingPassengers']

grouped = grouped.reset_index()
grouped

Смотрю на выбросы и думаю что делать с ними

In [ ]:
import seaborn as sns
sns.boxplot(grouped["PassengerDifference"])

Прихожу к выводу, что просто удалить выбросы я не могу, поэтому написала функцию, в которой выбросы заменяются на медиану

In [ ]:
def remove_outliers(group):
    median_ = group['PassengerDifference'].median()
    q1 = group['PassengerDifference'].quantile(0.25)
    q3 = group['PassengerDifference'].quantile(0.75)
    IQR = q3 - q1

    # границы для выбросов
    lower = q1 - 1.5 * IQR
    upper = q3 + 1.5 * IQR

    #выбросы на медиану
    group['PassengerDifference'] = group['PassengerDifference'].apply(lambda x: median_ if x < lower or x > upper else x)
    return group

grouped = grouped.reset_index(drop=True)
cleaned = (grouped.groupby('Year').apply(remove_outliers).reset_index(drop=True))

cleaned = cleaned.loc[:, ~cleaned.columns.str.contains('index')]

cleaned

In [ ]:
grouped = cleaned
grouped

Теперь встает вопрос о нормализации данных, пользуюсь min-max нормализацией, чтобы данные были в пределах 0 и 1 для дальнейшей оценки загруженности Почемц не zскор - у него нет границ, поэтому такая нормализация не даст нам нормальной картины

In [ ]:
def min_max_normalize(x):
    return (x - x.min()) / (x.max() - x.min())

grouped['PassengerDifference_norm'] = grouped.groupby(['Year', 'Quarter'])['PassengerDifference'].transform(min_max_normalize)
grouped

Для того чтобы оценить загруженность, пользуюсь pd.qcut, как раз разбивая на 5 примерно равных по количеству данных бинов

In [ ]:
res = grouped.copy()
res["StationLoad"] = pd.qcut(grouped['PassengerDifference_norm'], q=5, labels=False) + 1
#bin_edges = pd.qcut(grouped['PassengerDifference_minmax'], q=5, retbins=True)[1]
res

Оставляю только то что нужно

In [ ]:
result_data = res[['Year', 'Quarter', 'NameOfStation', 'Line', 'StationLoad']].copy()

result_data.head()

Теперь нужно добавить координаты для каждой стании метро, чтобы связать с нашим основным датасетом я обратилась к сайту Дадата, с помощью котрого спарсила все координаты для наших станций

In [ ]:
stations = result_data['NameOfStation'].unique()
stations

In [ ]:
import requests
import json
import pandas as pd

api_key = 'a40d2661860b1e9d8786a09ed73fd143cbde7066'
url = "http://suggestions.dadata.ru/suggestions/api/4_1/rs/suggest/metro"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Token {api_key}"
}

def get_station_coordinates(station_name):
    data = {
        "query": station_name
    }
    response = requests.post(url, headers=headers, data=json.dumps(data))

    if response.status_code == 200:
        result = response.json()
        if result["suggestions"]:
            station_data = result["suggestions"][0]["data"]
            return station_data["geo_lat"], station_data["geo_lon"]
        else:
            print(f"Станция {station_name} не найдена.")
            return None, None

#хранение координат
coordinates_df = pd.DataFrame(columns=["Station", "Latitude", "Longitude"])

for station in stations:
    lat, lon = get_station_coordinates(station)
    if lat and lon:
        new_row = pd.DataFrame({"Station": [station], "Latitude": [lat], "Longitude": [lon]})
        coordinates_df = pd.concat([coordinates_df, new_row], ignore_index=True)

coordinates_df

In [ ]:
df_merged = result_data.merge(coordinates_df, left_on='NameOfStation', right_on='Station', how='left')
df_merged

In [ ]:
df_merged.isna().sum()

In [ ]:
df_merged['NameOfStation'] = df_merged['NameOfStation'].apply(lambda x: '-'.join(word.capitalize() for word in x.split('-')))

new_data = df_merged[['Year', 'Quarter', 'NameOfStation', 'Line', 'StationLoad', 'Latitude', 'Longitude']].copy()

In [ ]:
new_data.head()

In [ ]:
#ready_df = pd.read_csv('main_df')

In [ ]:
metro = new_data[new_data['Year'] == 2024]

In [ ]:
metro

In [ ]:
metro_coords = metro[['Latitude', 'Longitude']].drop_duplicates()

In [ ]:
# Ищу район через координаты по геокодеру

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from tqdm import tqdm
import time

# Инициализация геокодера
geolocator = Nominatim(user_agent="moscow_schools")

def get_district(lat, lon, max_retries=3):
    """Получить район по координатам через Nominatim (OSM)."""
    retries = 0
    while retries < max_retries:
        try:
            time.sleep(1)  # Задержка между запросами
            location = geolocator.reverse((lat, lon), language="ru", exactly_one=True)
            if location and 'address' in location.raw:
                address = location.raw['address']
                district = address.get('suburb') or address.get('city_district') or address.get('borough')
                return district
            return None
        except (GeocoderTimedOut, GeocoderServiceError) as e:
            retries += 1
            print(f"Ошибка: {e}. Повторная попытка {retries}/{max_retries}...")
            time.sleep(10)  # Увеличиваем задержку при ошибке
    return None  # Если все попытки неудачны

# Применяем функцию к датафрейму
tqdm.pandas()
metro_coords['district'] = metro_coords.progress_apply(
    lambda row: get_district(row['Latitude'], row['Longitude']), axis=1
)

In [ ]:
metro.sample(5)

In [ ]:
# Для каких станций не определился район

metro_coords[metro_coords.district.isna()]

In [ ]:
# Присваиваю вручную значения станций метро


metro_coords.at[3426, "district"] = "район Внуково"
metro_coords.at[3505, "district"] = "район Хорошево-Мнёвники"
metro_coords.at[3563, "district"] = "район Коммунарка"
metro_coords.at[3576, "district"] = "район Коммунарка"
metro_coords.at[3604, "district"] = "район Коммунарка"
metro_coords.at[3612, "district"] = "район Внуково"
metro_coords.at[3615, "district"] = "район Внуково"
metro_coords.at[3625, "district"] = "район Коммунарка"
metro_coords.at[3678, "district"] = "район Коммунарка"
metro_coords.at[4202, "district"] = "район Коммунарка"
metro_coords.at[4403, "district"] = "район Коммунарка"
metro_coords.at[4408, "district"] = "район Коммунарка"

In [ ]:
# Проверяю на несовпадения названий районов метро и районов основого датасета
for d in metro_coords.district.unique():
    if d in ready_df.district_name.unique():
        continue
    else:
        print(d)

In [ ]:
sorted(ready_df.district_name.unique())

In [ ]:
to_rename = {
    'Бауманка': 'Басманный район',
    'Гавриково': 'район Южное Бутово',
    'Чернево': 'район Южное Бутово',
    'район Орехово-Борисово Южное': 'Орехово-Борисово Южное',
    'Кожухово': 'район Косино-Ухтомский',
    'Канатчиково': 'Донской район',
    'Дмитровский': 'Дмитровский район',
    'Черкизово':  'Молжаниновский район',
    'Павшинская Пойма': 'Митино',
    'Восход': 'район Левобережный',
    'Митино': 'район Митино',
    'район Хорошево-Мнёвники' : 'район Хорошёво-Мнёвники'
}


In [ ]:
# Переименовываю несовпадения в названиях районов для объединения

metro_coords.district = metro_coords.district.apply(lambda x: to_rename[x] if x in to_rename.keys() else x)

In [ ]:
# Проверяю на несовпадения названий районов метро и районов основого датасета
for d in metro_coords.district.unique():
    if d in ready_df.district_name.unique():
        continue
    else:
        print(d)

In [ ]:
metro_coords.sample(5)

In [ ]:
metro = pd.merge(metro, metro_coords, how='left', on=['Latitude', 'Longitude'])

In [ ]:
metro.head()

In [ ]:
df = pd.DataFrame(metro)

result_df = pd.DataFrame()

def add_dates_for_quarter(row):
    quarter = row['Quarter']
    if quarter == 'I квартал':
        date_range = pd.date_range(start='2024-01-01', end='2024-03-31 23:00:00', freq='h')
    elif quarter == 'II квартал':
        date_range = pd.date_range(start='2024-04-01', end='2024-06-30 23:00:00', freq='h')
    elif quarter == 'III квартал':
        date_range = pd.date_range(start='2024-07-01', end='2024-09-30 23:00:00', freq='h')
    elif quarter == 'IV квартал':
        date_range = pd.date_range(start='2024-10-01', end='2024-12-31 23:00:00', freq='h')

    time_df = pd.DataFrame(date_range, columns=['Timestamp'])

    for col in df.columns:
        time_df[col] = row[col]

    return time_df

for _, row in df.iterrows():
    result_df = pd.concat([result_df, add_dates_for_quarter(row)], ignore_index=True)

result_df

Добавила DayOfWeek, чтобы понимать, какой день недели

In [ ]:
result_df['DayOfWeek'] = pd.to_datetime(result_df['Timestamp']).dt.dayofweek

Теперь все праздники и каникулы

In [ ]:
holidays_2024 = [
    '2024-01-01',
    '2024-01-07',
    '2024-02-23',
    '2024-03-08',
    '2024-05-01',
    '2024-05-09',
    '2024-06-12',
    '2024-11-04'
]


new_year_holidays = pd.date_range(start='2023-12-30', end='2024-01-08').strftime('%Y-%m-%d').tolist()

defender_day = pd.date_range(start='2024-02-23', end='2024-02-25').strftime('%Y-%m-%d').tolist()

womens_day = pd.date_range(start='2024-03-08', end='2024-03-10').strftime('%Y-%m-%d').tolist()

may_first = pd.date_range(start='2024-04-28', end='2024-05-01').strftime('%Y-%m-%d').tolist()

may_victory = pd.date_range(start='2024-05-09', end='2024-05-12').strftime('%Y-%m-%d').tolist()


unity_day = pd.date_range(start='2024-11-03', end='2024-11-04').strftime('%Y-%m-%d').tolist()

# Объединяем все длинные выходные
long_weekends = new_year_holidays + defender_day + womens_day + may_first + may_victory  + unity_day

transferred_holidays = [
    '2024-05-10',
    '2024-12-31',
    '2024-04-29',
    '2024-04-30',
    '2024-12-30'
]


all_holidays = list(set(holidays_2024 + long_weekends + transferred_holidays))

# Сортируем даты для удобства
all_holidays.sort()
all_holidays


result_df['Date'] = result_df['Timestamp'].dt.date

all_holidays_date = [pd.to_datetime(date).date() for date in all_holidays]

result_df['IsHoliday'] = result_df['Date'].isin(all_holidays_date)
result_df['Time'] = result_df['Timestamp'].dt.time

result_df

Основная логика такая: буду отталкиваться от уже известной загруженности метро поквартально, при этом домнажаяя ее на соотв коэфы в зависомсти от текущих признаков Теперь основные зависимости распишу тут:

1) Зависимость от времени, если это утро или вечер, то это пиковая загруженность, поэтому коэф самый высокий. Если это день, то загруженность средняя, если ночь, то она совсем маленькая(и соотв коэф-ы)

2)День недели, обычно в будни загруженность метро выше чем в выходные

3)Праздники, в праздники загруженность ниже чем в будни

4)Также попробовала сюда прицепить сезонность, но из проверенного тут только то, что зимой загруженность метро куда больше чем летом

Мы в каждой части считаем фактор зависимости, потом их складываем и домножаем на наш базовый коэф-т

In [ ]:
def adjust_station_load(row):
    hour = row['Timestamp'].hour
    base_load = row['StationLoad']
    is_holiday = row['IsHoliday']
    day_of_week = row['Timestamp'].dayofweek
    month = row['Timestamp'].month

    #в зависимости от времени суток
    if 7 <= hour < 10:  # (пик)
        time_factor = 1.5
    elif 10 <= hour < 17:  # (средняя)
        time_factor = 1.0
    elif 17 <= hour < 21:  # (пик)
        time_factor = 1.5
    else:  #(низкая)
        time_factor = 0.5

    #день недели
    if day_of_week >= 5:  # Выходные
        day_factor = 0.8
    else:
        day_factor = 1.0  # Будни

    # праздники
    if is_holiday:
        holiday_factor = 0.7
    else:
        holiday_factor = 1.0

    #сезонность
    if month in [12, 1, 2]:  # Зима
        season_factor = 1.2
    elif month in [6, 7, 8]:  # Лето
        season_factor = 0.9
    else:
        season_factor = 1.0  # Другие месяцы

    # Итоговый коэффициент
    total_factor = time_factor * day_factor * holiday_factor * season_factor

    # Применяем коэффициент к базовой загруженности, где базовая загруженность - это поквартальная загруженность, от которой я отталкиваюсь
    adjusted_load = base_load * total_factor

    # Ограничиваем значения от 1 до 5
    return min(max(round(adjusted_load), 1), 5)

result_df['station_load'] = result_df.apply(adjust_station_load, axis=1)

In [ ]:
data_result = result_df[['Timestamp', 'Time', 'Quarter', 'NameOfStation', 'Date', 'Line', 'district','station_load', 'Latitude', 'Longitude']].copy()
data_result

In [ ]:
group_metro = data_result.groupby(['district', 'Date', 'Time'], as_index=False).station_load.sum()

In [ ]:
group_metro.sample(5)

Также добавила для каждого района количество станций, которые в него входят

In [ ]:
stat = data_result.groupby('district')['NameOfStation'].nunique().reset_index()
stat.columns = ['district', 'number_of_stations']

group_metro = group_metro.merge(stat, on='district', how='left')

group_metro

In [ ]:
group_metro.columns = ['district_name', 'date', 'time', 'station_load', 'number_of_stations']
group_metro

In [ ]:
# Добавлю квартал в основной датасет
def get_quarter(timestamp):
    month = timestamp.month
    if month in [1, 2, 3]:
        return "I квартал"
    elif month in [4, 5, 6]:
        return "II квартал"
    elif month in [7, 8, 9]:
        return "III квартал"
    else:
        return "IV квартал"

In [ ]:
ready_df['timestamp'] = pd.to_datetime(ready_df['timestamp'])

# Извлекаем время из timestamp и создаём новую колонку time
ready_df['time'] = ready_df['timestamp'].dt.time

ready_df['date'] = ready_df['timestamp'].dt.date

In [ ]:
ready_df['quarter'] = ready_df['timestamp'].apply(get_quarter)

In [ ]:
ready_df.sample(5)

In [ ]:
ready_df.shape

In [ ]:
ready_df_metro = pd.merge(ready_df, group_metro, how='left', on=['district_name', 'date', 'time'])

In [ ]:
ready_df_metro.shape

In [ ]:
ready_df_metro.sample(5)

In [ ]:
ready_df_metro.drop(columns=['time'], inplace=True)
ready_df_metro.drop(columns=['date'], inplace=True)

In [ ]:
ready_df_metro.info()

In [ ]:
ready_df_metro.station_load = ready_df_metro.station_load.fillna(0)

In [ ]:
#ready_df_metro.station_load_y = ready_df_metro.station_load_y.fillna(0)

In [ ]:
ready_df_metro.number_of_stations = ready_df_metro.number_of_stations.fillna(0)

In [ ]:
main_df = ready_df_metro.copy()
main_df

In [ ]:
#ready_df_metro.to_csv('main_df.csv', index=False)

## Развлекательные места в радиусе 5 км

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

In [ ]:
def create_dataframe_from_api(url):
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        if isinstance(data, list):
            df = data
        else:
            df = data.get('results', [])

        df = pd.DataFrame(df)

        return df
    else:
        print(f"Ошибка: {response.status_code}")
        time.sleep(60)
        return create_dataframe_from_api(url)


In [ ]:
coordinates = main_df[['lat', 'lon']].drop_duplicates()

In [ ]:
all_places_dfs = []

for index, row in coordinates.iterrows():
    lat = row['lat']
    lon = row['lon']

    url = (f"https://kudago.com/public-api/v1.4/places/?lang="
           f"&fields=title,short_title,subway,favorites_count,categories"
           f"&page=1"
           f"&page_size=300"
           f"&location=msk"
           f"&lat={lat}"
           f"&lon={lon}"
           f"&radius=5000")

    places_df = create_dataframe_from_api(url)

    if places_df is not None:
        places_df['lat'] = lat
        places_df['lon'] = lon

        all_places_dfs.append(places_df)

if all_places_dfs:
    final_places_df = pd.concat(all_places_dfs, ignore_index=True)
else:
    final_places_df = pd.DataFrame()

In [ ]:
final_places_df.groupby(['lat', 'lon']).title.count()

## Добавление информации о населении районов и доступности метро

Информация бралась с -  https://atlas.itdp.org/

In [ ]:
population = pd.read_csv('https://raw.githubusercontent.com/RuslanDavletov/Analysis-of-taxi-demand/main/data/population_and_metro_accessibility.csv')
population.head()

In [ ]:
population = population.drop('Unnamed: 0', axis=1)

In [ ]:
# Преобразуем данные о населении в целочисленный формат
population['city_popdensitytotal_2023'] = population['city_popdensitytotal_2023'].astype('int')

In [ ]:
# Добавляем информацию в основной датасет
df = main_df.merge(population,how='left', left_on='district_name', right_on='name')
df

In [ ]:
# Удаляем ненужные поля
df = df.drop(['name', 'Unnamed: 1'], axis=1)

# Переименовываем добавленные поля
df = df.rename(columns={'city_popdensitytotal_2023' : 'population_district_2023', 
                        'People Near Rapid Transport': 'population_near_metro_percent'})

In [ ]:
# Проверяем данные на дубликаты
df.duplicated().sum()

In [ ]:
# Удаляем найденные дубликаты
df = df.drop_duplicates()
df

In [ ]:
# Проверяем данные на пропуски
df.isna().sum()

In [ ]:
main_df = df

In [ ]:
main_df

## Цены на недвижимость по районам Москвы

In [ ]:
#данные из парсера
estate_price = pd.read_csv('https://raw.githubusercontent.com/RuslanDavletov/Analysis-of-taxi-demand/main/data/moscow_estate_price_raw.csv')

In [ ]:
#таблица со всеми районами из основного файла с поездками
districts = pd.DataFrame(main_df['district_name'].unique(), columns=['district_name'])

In [ ]:
#проверяем тип данных 
estate_price.info()

In [ ]:
# для дальнейшего анализа будем использовать данные о стоимости жилья только на вторичном рынке 
estate_price=estate_price[['Район','Вторичный рынок']]

In [ ]:
#добавляем колонку, где будут только сами названия,без типа пункта
estate_price.loc[:, 'estate_name_clean'] = estate_price['Район'].replace(
    to_replace = ['[Рр]айон','поселение','городской округ'], 
    value ='', 
    regex = True
).str.strip(to_strip=None)

In [ ]:
#тоже самое делаем в таблице districts
districts.loc[:, 'main_name_clean'] = districts['district_name'].replace(
    to_replace = ['[Рр]айон','поселение','городской округ'], 
    value ='', 
    regex = True
).str.strip(to_strip=None)

In [ ]:
#districts['district_name'].unique()

In [ ]:
#estate_price['Район'].unique()

In [ ]:
#мерджим таблицу districts и estate_price по новым колонкам с названиями
merged_df = districts.merge(estate_price, how='left', left_on='main_name_clean', right_on='estate_name_clean')

In [ ]:
#проверяем наличие пропусков
merged_df[merged_df['Вторичный рынок'].isna()]

Пропуски возникли из-за териториальных изменений Москвы в 2024 году:в таблице districts актуальные данные по районам, в таблице estate_price- устаревшие, обработаем эти пропуски вручную,используя также открытые источники (https://www.irn.ru/kvartiry/moskva/ceny-po-rayonam/?ysclid=m82cab41ua433689328)

In [ ]:
#цены в р.Коммунарка совпадают с ценами в р.Южное Бутово 
merged_df.loc[merged_df['district_name'] == 'район Коммунарка', 'Вторичный рынок'] = \
    estate_price.loc[estate_price['estate_name_clean'] == 'Южное Бутово', 'Вторичный рынок'].values[0]

#в таблице estate_price районы Филимонковский,Внуково,Вороново,Краснопахорски обозначены как поселения 
#(Филимонковское,Внуковское,Вороновское,Краснопахорское) - окончания не совпадают,поэтому появились пропуски
merged_df.loc[
    merged_df['district_name'].isin({'Филимонковский район', 'район Внуково', 'район Вороново', 'Краснопахорский район'}), 
    'Вторичный рынок'
] = merged_df['district_name'] \
    .replace({
        'Филимонковский район': 'Филимонковское',
        'район Внуково': 'Внуковское',
        'район Вороново': 'Вороновское',
        'Краснопахорский район': 'Краснопахорское'
    }) \
    .map(estate_price.set_index('estate_name_clean')['Вторичный рынок'])

#район Бекасово соответствует упраздненным поселениям Новофёдоровское и Киевский - возьмем среднее от их цен 
merged_df.loc[merged_df['district_name'] == 'район Бекасово', 'Вторичный рынок'] = \
    estate_price.loc[estate_price['estate_name_clean'].isin(['Новофёдоровское', 'Киевский']), 'Вторичный рынок'].mean()

In [ ]:
#проверяем наличие пропусков после обработки
merged_df[merged_df['Вторичный рынок'].isna()]

In [ ]:
#дополнительно проверяем цены
merged_df[merged_df['Вторичный рынок'] ==  0]

In [ ]:
#Силино относится к Зеленоградску,цены соответствуют Матушкино,Савелки и тд
merged_df.loc[merged_df['district_name'] == 'район Силино', 'Вторичный рынок'] = \
    estate_price.loc[estate_price['estate_name_clean'] == 'Матушкино', 'Вторичный рынок'].values[0]
#проверяем еще раз
merged_df[merged_df['Вторичный рынок'] ==  0]

In [ ]:
#в финальной таблице сохраняем названия из main_df и соответствующие цены
msc_estate_price_final = merged_df[['district_name','Вторичный рынок']].rename(columns={'Вторичный рынок': 'price_m2'})

In [ ]:
#сохраняем таблицу
#msc_estate_price_final.to_csv("msc_estate_price_final.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Объединяем данные по столбцу 'district_name', добавляя столбец 'price_m2'
main_df = main_df.merge(msc_estate_price_final[['district_name', 'price_m2']], on='district_name', how='left')

## Описание полей (неполное)

### Описание полей DataFrame о поездках такси и каршеринга по районам Москвы

- **`timestamp` (datetime64[ns])** — Метка времени, соответствующая периоду наблюдений.  
- **`district_name` (object)** — Название района Москвы.  
- **`district_id` (int64)** — Уникальный идентификатор района.  
- **`lat` (float64)** — Широта центра района.  
- **`lon` (float64)** — Долгота центра района.  
- **`n_taxi_start` (float64)** — Количество поездок на такси, начавшихся в районе.  
- **`n_taxi_end` (float64)** — Количество поездок на такси, завершившихся в районе.  
- **`n_carsharing_start` (float64)** — Количество поездок на каршеринге, начавшихся в районе.  
- **`n_carsharing_end` (float64)** — Количество поездок на каршеринге, завершившихся в районе.  
- **`temperature` (float64)** — Температура воздуха (°C) на момент наблюдений.  
- **`precipitation` (float64)** — Количество осадков (мм) на момент наблюдений.  
- **`humidity` (float64)** — Влажность воздуха (%) на момент наблюдений.  
- **`wind_speed` (float64)** — Скорость ветра (м/с) на момент наблюдений.  
- **`weather_code` (float64)** — Код типа погоды согласно метеоданным

Код состояния погоды (**weather_code**) в принимает значения от **1 до 25**, а также **NaN** для отсутствующих данных. Эти значения соответствуют различным погодным явлениям, таким как типы осадков, облачность, видимость и другие метеорологические явления:

| Код | Погодное условие         |
|-----|--------------------------|
| 1   | Ясно                    |
| 2   | Солнечно                 |
| 3   | Облачно                  |
| 4   | Пасмурно                 |
| 5   | Туман                    |
| 6   | Ледяной туман            |
| 7   | Легкий дождь             |
| 8   | Дождь                    |
| 9   | Сильный дождь            |
| 10  | Ледяной дождь            |
| 11  | Сильный ледяной дождь    |
| 12  | Дождь со снегом          |
| 13  | Сильный дождь со снегом  |
| 14  | Легкий снегопад          |
| 15  | Снегопад                 |
| 16  | Сильный снегопад         |
| 17  | Дождевой ливень          |
| 18  | Сильный дождевой ливень  |
| 19  | Дождь со снегом (ливень) |
| 20  | Сильный дождь со снегом  |
| 21  | Снежный ливень           |
| 22  | Сильный снежный ливень   |
| 23  | Молния                   |
| 24  | Град                     |
| 25  | Гроза                    |
| **NaN** | Нет данных           |
